# FAS Preliminary Results: Phonemic Verbal Fluency Scoring

This notebook scores participant responses for the FAS phonemic verbal fluency task
(words beginning with F, A, S) using count-based metrics following Tallberg et al. (2008)
and Borkowski et al. (1967).

**Dataset:** FAS test (F, A, S letters, 60 s each), 4 diagnostic groups (HC, MCI, non-AD, AD)

**Key metrics:**
- `total_fas_score` — sum of valid words across F + A + S (main clinical outcome)
- `valid_f / valid_a / valid_s` — valid word count per letter (excluding proper nouns and repetitions)
- `letter_asymmetry` — std of [valid_f, valid_a, valid_s] (uneven performance across letters)
- `proper_nouns_count` / `repetitions_count` — error types
- `mean_edit_distance_f/a/s` — exploratory: normalised Levenshtein distance between consecutive words (phonemic similarity)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from thesis_project.preprocessing.data_loader import FAS_PATH, load_fas_data
from thesis_project.scoring.fas_scorer import score_fas

warnings.filterwarnings('ignore')

# Paths
DATA_PATH = Path('../data/xlsx/FAS-syntheticData_v1.xlsx')
OUTPUT_PATH = Path('../data/processed/fas_scored_results.csv')

print('Setup complete.')

## 1. Load FAS Data

In [ ]:
participants = load_fas_data(DATA_PATH)

print(f'Loaded {len(participants)} participants')

diagnoses = pd.Series([p['diagnosis'] for p in participants])
print(f'\nDiagnosis distribution:')
print(diagnoses.value_counts().to_string())

total_flagged = sum(len(p['flagged_errors']) for p in participants)
print(f'\nTotal flagged errors (proper nouns / rule violations): {total_flagged}')

# Preview a couple of participants
print(f'\nExample participants:')
for p in participants[:2]:
    n_f = len([w for w in p['responses_f'] if w])
    n_a = len([w for w in p['responses_a'] if w])
    n_s = len([w for w in p['responses_s'] if w])
    print(f"  {p['participant_id']} ({p['diagnosis']}): F={n_f} words, A={n_a} words, S={n_s} words")
    print(f"    F: {p['responses_f'][:5]} ...")
    if p['flagged_errors']:
        print(f"    Flagged: {p['flagged_errors']}")

## 2. Score FAS Responses

FAS scoring is count-based — no embeddings required.

In [ ]:
records = []
for p in participants:
    metrics = score_fas(
        responses_f=p['responses_f'],
        responses_a=p['responses_a'],
        responses_s=p['responses_s'],
        flagged_errors=p['flagged_errors'],
    )
    records.append({
        'participant_id': p['participant_id'],
        'diagnosis': p['diagnosis'],
        'age': p['age'],
        'gender': p['gender'],
        **metrics,
    })

scored = pd.DataFrame(records)
print(f'Scoring complete. Shape: {scored.shape}')
print(f'\ntotal_fas_score — min={scored["total_fas_score"].min()}, '
      f'max={scored["total_fas_score"].max()}, '
      f'mean={scored["total_fas_score"].mean():.1f}')
print(f'\nColumns: {scored.columns.tolist()}')

## 3. Results

### 3a. Summary Table: FAS Scores by Diagnostic Group

In [ ]:
diag_order = ['HC', 'MCI', 'non-AD', 'AD']

summary_rows = []
for diag in diag_order:
    sub = scored[scored['diagnosis'] == diag]
    if sub.empty:
        continue
    summary_rows.append({
        'Diagnosis': diag,
        'N': len(sub),
        'TotalFAS_mean': sub['total_fas_score'].mean(),
        'TotalFAS_std': sub['total_fas_score'].std(),
        'ValidF_mean': sub['valid_f'].mean(),
        'ValidA_mean': sub['valid_a'].mean(),
        'ValidS_mean': sub['valid_s'].mean(),
        'Asymmetry_mean': sub['letter_asymmetry'].mean(),
        'ProperNouns_mean': sub['proper_nouns_count'].mean(),
        'Repetitions_mean': sub['repetitions_count'].mean(),
    })

summary_df = pd.DataFrame(summary_rows)
print('Summary: FAS Metrics by Diagnostic Group')
print('=' * 90)
print(summary_df.to_string(index=False, float_format='%.2f'))
print()
print('TotalFAS     = valid_f + valid_a + valid_s (primary clinical outcome)')
print('Asymmetry    = std([valid_f, valid_a, valid_s]) — uneven performance across letters')
print('ProperNouns  = words flagged as angle-bracket errors, excluded from valid count')

### 3b. Box Plot: Total FAS Score by Diagnostic Group

In [ ]:
colors = ['#2ecc71', '#f39c12', '#9b59b6', '#e74c3c']  # HC, MCI, non-AD, AD

fig, ax = plt.subplots(figsize=(10, 6))

data = [scored[scored['diagnosis'] == d]['total_fas_score'].values for d in diag_order]
bp = ax.boxplot(data, labels=diag_order, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

means = [np.mean(d) for d in data]
ax.scatter(range(1, len(diag_order)+1), means, color='black', marker='D', s=60, zorder=3, label='Mean')

ax.set_ylabel('Total FAS Score (valid words F+A+S)', fontsize=12)
ax.set_xlabel('Diagnostic Group', fontsize=12)
ax.set_title('Distribution of Total FAS Scores by Diagnostic Group', fontsize=14)
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/fas_boxplot_total.png', dpi=150)
plt.show()

print('Boxplot saved to data/processed/fas_boxplot_total.png')

### 3c. Per-Letter Breakdown: Valid Words for F, A, S by Diagnostic Group

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)

for ax, (letter, col) in zip(axes, [('F', 'valid_f'), ('A', 'valid_a'), ('S', 'valid_s')]):
    data = [scored[scored['diagnosis'] == d][col].values for d in diag_order]
    bp = ax.boxplot(data, labels=diag_order, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    means = [np.mean(d) for d in data]
    ax.scatter(range(1, len(diag_order)+1), means, color='black', marker='D', s=50, zorder=3)
    ax.set_ylabel(f'Valid Words — Letter {letter}', fontsize=11)
    ax.set_xlabel('Diagnostic Group', fontsize=11)
    ax.set_title(f'Letter {letter}', fontsize=13)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Valid Word Counts per Letter by Diagnostic Group', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/fas_boxplot_per_letter.png', dpi=150)
plt.show()

print('Per-letter boxplot saved to data/processed/fas_boxplot_per_letter.png')

### 3d. Error Analysis: Proper Nouns, Repetitions, and Letter Asymmetry

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = [
    ('proper_nouns_count', 'Proper Noun Errors'),
    ('repetitions_count', 'Repetitions'),
    ('letter_asymmetry', 'Letter Asymmetry (std of F/A/S)'),
]

for ax, (col, label) in zip(axes, metrics_to_plot):
    data = [scored[scored['diagnosis'] == d][col].dropna().values for d in diag_order]
    bp = ax.boxplot(data, labels=diag_order, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    means = [np.nanmean(d) for d in data]
    ax.scatter(range(1, len(diag_order)+1), means, color='black', marker='D', s=50, zorder=3)
    ax.set_ylabel(label, fontsize=11)
    ax.set_xlabel('Diagnostic Group', fontsize=11)
    ax.set_title(label, fontsize=12)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('FAS Error Metrics by Diagnostic Group', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/fas_error_metrics.png', dpi=150)
plt.show()

print('Error metrics plot saved to data/processed/fas_error_metrics.png')

### 3e. Exploratory: Phonemic Similarity Between Consecutive Words (Edit Distance)

In [ ]:
# mean_edit_distance = mean normalised Levenshtein between consecutive responses.
# Lower = phonemically similar consecutive words (phonemic clustering / perseveration)
# Higher = more phonemically varied consecutive words

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (letter, col) in zip(axes, [('F', 'mean_edit_distance_f'), ('A', 'mean_edit_distance_a'), ('S', 'mean_edit_distance_s')]):
    data = [scored[scored['diagnosis'] == d][col].dropna().values for d in diag_order]
    bp = ax.boxplot(data, labels=diag_order, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    means = [np.nanmean(d) for d in data]
    ax.scatter(range(1, len(diag_order)+1), means, color='black', marker='D', s=50, zorder=3)
    ax.set_ylabel('Mean Normalised Edit Distance', fontsize=11)
    ax.set_xlabel('Diagnostic Group', fontsize=11)
    ax.set_title(f'Letter {letter}: Consecutive Edit Distance\n(lower = phonemically similar consecutive words)', fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Phonemic Similarity Between Consecutive Words (per letter)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/fas_edit_distance.png', dpi=150)
plt.show()

print('Edit distance plot saved to data/processed/fas_edit_distance.png')

# Summary
print('\nMean edit distance across diagnostic groups:')
for diag in diag_order:
    sub = scored[scored['diagnosis'] == diag]
    f_ed = sub['mean_edit_distance_f'].mean()
    a_ed = sub['mean_edit_distance_a'].mean()
    s_ed = sub['mean_edit_distance_s'].mean()
    print(f'  {diag}: F={f_ed:.3f}, A={a_ed:.3f}, S={s_ed:.3f}')

### 3f. Score Distribution: Total FAS Score Across All Participants

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for diag, color in zip(diag_order, colors):
    sub = scored[scored['diagnosis'] == diag]
    ax.hist(
        sub['total_fas_score'],
        bins=15,
        alpha=0.5,
        color=color,
        label=f'{diag} (n={len(sub)}, mean={sub["total_fas_score"].mean():.1f})',
        edgecolor='white',
    )

ax.set_xlabel('Total FAS Score (valid F+A+S)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Total FAS Scores by Diagnostic Group', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/fas_score_distribution.png', dpi=150)
plt.show()

print('Score distribution saved to data/processed/fas_score_distribution.png')

## 4. Save Results

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output_cols = [
    'participant_id', 'diagnosis', 'age', 'gender',
    'total_f', 'total_a', 'total_s',
    'valid_f', 'valid_a', 'valid_s',
    'total_fas_score',
    'proper_nouns_count', 'repetitions_count',
    'letter_asymmetry',
]
scored[output_cols].to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(scored)} participant records to {OUTPUT_PATH}')

summary_path = OUTPUT_PATH.parent / 'fas_summary_by_diagnosis.csv'
summary_df.to_csv(summary_path, index=False)
print(f'Saved summary table to {summary_path}')

## Key Takeaways

1. **Total FAS score**: The primary clinical outcome. Expect a gradient HC > MCI > non-AD > AD reflecting declining phonemic fluency with dementia severity.

2. **Per-letter variation**: F, A, and S differ in difficulty (F tends to yield fewer valid words in Swedish). The per-letter breakdown reveals whether group differences are uniform or letter-specific.

3. **Letter asymmetry**: Higher asymmetry (std across F/A/S) may indicate strategy breakdown — patients who perform well on one letter but poorly on others.

4. **Error rates**: Proper nouns and repetitions are excluded from valid counts (following Tallberg et al.). Their frequency by diagnostic group can reveal qualitatively different error patterns.

5. **Edit distance** (exploratory): Low normalised Levenshtein distance between consecutive words within a letter suggests phonemic clustering (e.g., 'fart' → 'fara' → 'fast'). Higher distance suggests more varied word selection. This metric is not yet validated for Swedish FAS.